# Pilot: does INT8 quantization hurt trust more on field photos?

MobileNetV3-Large, FP32 vs INT8 post-training quantization, PlantVillage (lab) -> PlantDoc (field).

**Run the cells top to bottom.** Works on your GPU laptop (Windows or Linux) or on Google Colab.

| Step | What it does | Time |
| --- | --- | --- |
| 1 | Settings | - |
| 2 | Check GPU and packages | 1 min |
| 3 | Get the code from your GitHub repo | 1 min |
| 4-5 | Download PlantDoc and PlantVillage | 10-30 min (once) |
| 6 | Check the folders | 1 min |
| 7 | Quick test run | ~3 min |
| 8 | Full pilot run | 45-90 min |
| 9 | Show results and the GO / NO-GO verdict | - |
| 10 | Save results into the repo for GitHub | 1 min |

Requirements on a laptop: **git** installed, and PyTorch with CUDA installed from https://pytorch.org (step 2 checks this).

## 1. Settings
Change these if needed, then run the cell.

In [ ]:
from pathlib import Path

# Where everything (code + data + runs) is stored. Needs ~4 GB free.
WORKDIR = Path.home() / "crop-pilot"

# How to get PlantVillage:
#   "github" - sparse clone of only the colour images from spMohanty/PlantVillage-Dataset (no account needed)
#   "kaggle" - Kaggle API (needs ~/.kaggle/kaggle.json)
#   "manual" - you already have it: set PV_MANUAL_PATH to the folder that has one sub-folder per class
PV_SOURCE = "github"
PV_MANUAL_PATH = None          # e.g. r"D:\data\plantvillage\color"

SEED = 0
RUN = "v2"                     # name of this experiment version; results go to runs/pilot_<RUN>_seed<SEED>
WORKERS = 2                    # data-loading processes; 2 is safe on Windows laptops, try 4 on Linux/Colab
BATCH = 64                     # lower to 32 if you get a CUDA out-of-memory error on a 6 GB GPU
PRETRAINED = True              # start from ImageNet weights (downloaded once, ~20 MB). Keep True for real runs.

REPO_URL = "https://github.com/saurabh-chy/quantization-trust-crop-disease.git"
WORKDIR.mkdir(parents=True, exist_ok=True)
EXTRA = [] if PRETRAINED else ["--no-pretrained"]
print("Working folder:", WORKDIR)

Helper to run shell commands and stream their output into the notebook (works the same on Windows, Linux and Colab).

In [ ]:
import subprocess, sys, os, shutil

def run(cmd, cwd=None, check=True):
    """Run a command (list of args), printing output live."""
    print(">", " ".join(str(c) for c in cmd))
    p = subprocess.Popen([str(c) for c in cmd], cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {p.returncode}")
    return p.returncode

# Find git even if Jupyter's PATH hasn't picked it up yet (common on Windows right after installing git)
if not shutil.which("git"):
    for d in [r"C:\Program Files\Git\cmd", r"C:\Program Files (x86)\Git\cmd",
              os.path.expanduser(r"~\AppData\Local\Programs\Git\cmd")]:
        if os.path.exists(os.path.join(d, "git.exe")):
            os.environ["PATH"] = d + os.pathsep + os.environ["PATH"]
            break
import stat, time

def rm_tree(p):
    """Delete a folder, including git's read-only files on Windows."""
    def fix(func, path, _):
        os.chmod(path, stat.S_IWRITE); func(path)
    if Path(p).exists():
        shutil.rmtree(p, onerror=fix)

def clone(args, dest, tries=3, after=None):
    """git clone with retries; removes half-finished copies first. `after` runs extra git commands in dest."""
    for i in range(1, tries + 1):
        rm_tree(dest)
        ok = run(["git", "clone", *args, dest], check=False) == 0
        if ok and after:
            ok = all(run(c, cwd=dest, check=False) == 0 for c in after)
        if ok:
            return
        print(f"Attempt {i} failed; retrying in 10 s...")
        time.sleep(10)
    raise RuntimeError("Download failed. Scroll up for git's message and send it to Claude.")

import hashlib, re

def safe_name(name, maxlen=80):
    """Windows-safe file name: replace forbidden characters, shorten very long names."""
    stem, dot, ext = name.rpartition(".")
    if not dot or len(ext) > 5:
        stem, ext = name, ""
    clean = re.sub(r'[<>:"/\\|?*\x00-\x1f%&=]', "_", stem).strip(" .")
    if len(clean) > maxlen or clean != stem:
        clean = clean[:maxlen].rstrip(" ._") + "_" + hashlib.md5(name.encode()).hexdigest()[:8]
    return clean + ("." + ext.lower() if ext else "")

def fetch_git_files(url, dest, prefixes=("train/", "test/"), tries=3):
    """Clone without checkout, then write each file with a Windows-safe name.
    Avoids git failing on file names Windows cannot create."""
    tmp = Path(str(dest) + "_git")
    clone(["--depth", "1", "--no-checkout", url], tmp, tries=tries)
    listing = subprocess.run(["git", "ls-tree", "-r", "-z", "HEAD"], cwd=tmp,
                             capture_output=True, check=True).stdout.decode("utf-8", "replace")
    entries = []
    for rec in listing.split("\0"):
        if not rec:
            continue
        meta, path = rec.split("\t", 1)
        mode, kind, sha = meta.split()
        if kind == "blob" and path.startswith(prefixes):
            entries.append((sha, path))
    print(f"Writing {len(entries)} files with Windows-safe names...")
    cat = subprocess.Popen(["git", "cat-file", "--batch"], cwd=tmp,
                           stdin=subprocess.PIPE, stdout=subprocess.PIPE)
    renamed = 0
    for i, (sha, path) in enumerate(entries, 1):
        cat.stdin.write((sha + "\n").encode()); cat.stdin.flush()
        header = cat.stdout.readline().split()
        size = int(header[2])
        data = cat.stdout.read(size); cat.stdout.read(1)
        parts = path.split("/")
        new = safe_name(parts[-1])
        renamed += new != parts[-1]
        out = Path(dest).joinpath(*parts[:-1], new)
        out.parent.mkdir(parents=True, exist_ok=True)
        out.write_bytes(data)
        if i % 500 == 0:
            print(f"  {i}/{len(entries)}")
    cat.stdin.close(); cat.wait()
    rm_tree(tmp)
    print(f"Done: {len(entries)} files, {renamed} renamed for Windows.")

assert shutil.which("git"), ("git not found. Install it from https://git-scm.com, or open Command Prompt, "
                             "run  where git  and add that folder to PATH.")
print("git found:", shutil.which("git"))

## 2. Check GPU and packages

In [ ]:
try:
    import torch, torchvision
except ImportError:
    raise SystemExit(
        "PyTorch is not installed in this kernel.\n"
        "Install it with the command for your CUDA version from https://pytorch.org (Get Started), "
        "e.g.  pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121\n"
        "then restart the kernel and run again.")

print("torch", torch.__version__, "| torchvision", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
else:
    print("WARNING: no GPU visible. Training will be very slow. Check your NVIDIA driver and the PyTorch CUDA build.")

# The other, lightweight packages
run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "matplotlib", "pillow"])

## 3. Get the code from your GitHub repository

In [ ]:
CODE = WORKDIR / "quantization-trust-crop-disease"
if (CODE / ".git").exists():
    run(["git", "pull"], cwd=CODE)
else:
    clone([REPO_URL], CODE)
assert (CODE / "pilot.py").exists(), "pilot.py not found in the repo"
print("Code ready at", CODE)

## 4. Download PlantDoc (field images)

In [ ]:
DATA = WORKDIR / "data"
DATA.mkdir(exist_ok=True)
PD = DATA / "PlantDoc-Dataset"
if not (PD / "train").exists():
    # PlantDoc has file names Windows cannot create (they contain '?'), so a normal
    # git clone fails on Windows. This writes every image with a safe name instead.
    rm_tree(PD)
    fetch_git_files("https://github.com/pratikkayal/PlantDoc-Dataset", PD)
print("PlantDoc ready:", PD, "| sub-folders:", sorted(p.name for p in PD.iterdir() if p.is_dir()))

## 5. Download PlantVillage (lab images)
With `PV_SOURCE = "github"` this fetches only the colour images (about 1 GB of the repository). It can take 10-30 minutes; let it run.

In [ ]:
def find_class_root(root):
    """Find the folder whose sub-folders are PlantVillage classes (names contain '___')."""
    root = Path(root)
    cands = [root] + [p for p in root.rglob("*") if p.is_dir()]
    for c in cands:
        subs = [s for s in c.iterdir() if s.is_dir()] if c.exists() else []
        if sum("___" in s.name for s in subs) >= 10:
            if "color" in c.name.lower() or not any("color" in p.name.lower() for p in cands):
                return c
    for c in cands:   # fallback: any folder with many class-like sub-folders
        subs = [s for s in c.iterdir() if s.is_dir()]
        if sum("___" in s.name for s in subs) >= 10:
            return c
    return None

if PV_SOURCE == "manual":
    PV = Path(PV_MANUAL_PATH)

elif PV_SOURCE == "github":
    repo = DATA / "PlantVillage-Dataset"
    if not (repo / "raw" / "color").exists():
        clone(["--depth", "1", "--filter=blob:none", "--sparse",
               "https://github.com/spMohanty/PlantVillage-Dataset"], repo,
              after=[["git", "sparse-checkout", "set", "raw/color"]])
    PV = repo / "raw" / "color"

elif PV_SOURCE == "kaggle":
    target = DATA / "plantvillage_kaggle"
    if not target.exists():
        run([sys.executable, "-m", "pip", "install", "-q", "kaggle"])
        run(["kaggle", "datasets", "download", "-d", "abdallahalidev/plantvillage-dataset",
             "-p", target, "--unzip"])
    PV = find_class_root(target)
else:
    raise ValueError("PV_SOURCE must be 'github', 'kaggle' or 'manual'")

assert PV and Path(PV).exists(), f"PlantVillage class folder not found ({PV}). Set PV_SOURCE='manual' and PV_MANUAL_PATH."
print("PlantVillage ready:", PV)

## 6. Check the folders

In [ ]:
IMG = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
def count(d): return sum(1 for p in Path(d).rglob("*") if p.suffix.lower() in IMG)

pv_classes = sorted(p.name for p in Path(PV).iterdir() if p.is_dir())
print(f"PlantVillage: {len(pv_classes)} classes, {count(PV)} images")
for split in ("train", "test"):
    d = PD / split
    print(f"PlantDoc {split}: {len([p for p in d.iterdir() if p.is_dir()])} classes, {count(d)} images")
# Expect about 38 PlantVillage classes (~54k images) and 27-28 PlantDoc classes (~2.5k images).

## 7. Quick test run (about 3 minutes)
Checks paths, class matching and the whole pipeline on a small sample. **Look at the first lines of the output:** if it prints `[!] Mapping entries not found`, copy that block and send it to Claude so the class mapping can be fixed. A few unmatched entries are fine; the run stops itself if fewer than 10 classes match.

In [ ]:
TEST_OUT = WORKDIR / "runs" / "test"
shutil.rmtree(TEST_OUT, ignore_errors=True)
run([sys.executable, CODE / "pilot.py", "--pv", PV, "--pd", PD, "--out", TEST_OUT,
     "--epochs", 1, "--max-per-class", 50, "--boot", 100,
     "--workers", WORKERS, "--bs", BATCH, "--seed", SEED] + EXTRA)

## 8. Full pilot run (about 45-90 minutes on a 6 GB GPU)
Keep the laptop plugged in. If it stops halfway, just run the cell again: finished stages (trained model, logits) are reused.

In [ ]:
OUT = WORKDIR / "runs" / f"pilot_{RUN}_seed{SEED}"
CMD = [sys.executable, CODE / "pilot.py", "--pv", PV, "--pd", PD, "--out", OUT,
       "--workers", WORKERS, "--bs", BATCH, "--seed", SEED] + EXTRA
run(CMD)

## 9. Results

In [ ]:
import json
from IPython.display import Image, display

res = json.load(open(OUT / "results.json"))
print(f"Shared classes: {res['n_classes']} | lab train: {res.get('n_pv_train','?')} | lab test: {res['n_pv_test']} "
      f"| field-cal: {res.get('n_pd_cal','-')} | field test: {res['n_pd']}")
print("INT8 mode:", res["quant_mode"], "| variants:", ", ".join(res.get("variants", ["INT8"])))
print("Temperatures:", res.get("T"), "\n")

cols = ["acc", "macro_f1", "ece", "aurc", "acc_at_80", "err_auroc", "conf_wrong"]
print(f"{'set':<8} {'model':<22}" + "".join(f"{c:>11}" for c in cols))
for r in res["table"]:
    print(f"{r['dataset']:<8} {r['model']:<22}" + "".join(f"{r[c]:>11.4f}" for c in cols))

gaps = res["gap"] if isinstance(next(iter(res["gap"].values())), dict) else {"INT8": res["gap"]}
flips = res["flip"] if "pd" not in res["flip"] else {"INT8": res["flip"]}
hints = res["hint"] if isinstance(res["hint"], dict) else {"INT8": res["hint"]}
for name in gaps:
    print(f"\n== {name} ==")
    print("Shift gap = (quantized - FP32 on field) - (quantized - FP32 on lab), 95% CI:")
    for k, (m, lo, hi) in gaps[name].items():
        print(f"  {k:>10}: {m:+.4f}  [{lo:+.4f}, {hi:+.4f}]")
    print("Flip rate (quantized changes the FP32 answer), by FP32 confidence:")
    for ds, v in flips[name].items():
        print(f"  {ds:<8} overall {v['overall']:.4f} | " +
              "  ".join(f"[{b}] n={n} {f:.3f}" for b, n, f in v["bins"]))
    print("VERDICT:", hints[name])
display(Image(filename=str(OUT / "pilot.png")))

**How to read the verdict**
There is one verdict per variant (INT8 and INT4-W):
- **GO**: that precision hurts reliability more on field photos and the interval excludes zero.
- **LEAN GO**: the effect points the right way but the interval includes zero. Run seeds 1 and 2 (next cell) before deciding.
- **NULL**: that precision does not widen the lab-to-field gap.

Send `results.json` and `pilot.png` to Claude to interpret.

### Optional: extra seeds (for LEAN GO)
Each seed is another full run (45-90 min). Set `RUN_EXTRA = True` to run them.

In [ ]:
RUN_EXTRA = False
if RUN_EXTRA:
    for s in (1, 2):
        run([sys.executable, CODE / "pilot.py", "--pv", PV, "--pd", PD,
             "--out", WORKDIR / "runs" / f"pilot_{RUN}_seed{s}",
             "--workers", WORKERS, "--bs", BATCH, "--seed", s] + EXTRA)
        r = json.load(open(WORKDIR / "runs" / f"pilot_{RUN}_seed{s}" / "results.json"))
        print(f"seed {s}:", r["hint"])

## 10. Save results into the repository
Copies each finished run's metrics, plot, class list and raw logits into `results/` (model weights are left out on purpose; they are too large for git).
Then either push from here (needs your GitHub login on this machine), or send `results.json` and `pilot.png` to Claude to commit for you.

In [ ]:
saved = []
for run_dir in sorted((WORKDIR / "runs").glob("pilot_*seed*")):
    if not (run_dir / "results.json").exists():
        continue
    dest = CODE / "results" / run_dir.name
    dest.mkdir(parents=True, exist_ok=True)
    for f in ("results.json", "pilot.png", "classes.json", "config.json", "logits.npz"):
        if (run_dir / f).exists():
            shutil.copy2(run_dir / f, dest / f)
    seed = run_dir.name.rsplit("seed", 1)[1]
    (dest / "command.txt").write_text(
        f"python pilot.py --pv <PlantVillage color folder> --pd <PlantDoc-Dataset> "
        f"--out runs/{run_dir.name} --workers {WORKERS} --bs {BATCH} --seed {seed}\n"
        f"torch {torch.__version__}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}\n")
    saved.append(dest)
print("Saved:", *saved, sep="\n  ")

In [ ]:
PUSH = False   # set True to commit and push from this machine
if PUSH:
    run(["git", "add", "results"], cwd=CODE)
    run(["git", "commit", "-m", f"Add pilot results ({', '.join(p.name for p in saved)})"], cwd=CODE)
    run(["git", "push"], cwd=CODE)   # asks for your GitHub login the first time